# 02 - Regresión Ridge (Penalización $L_2$)

## Curso de Machine Learning: Modelos de Regresion

En este cuaderno estudiaremos en detalle la **Regresion Ridge** (tambien conocida como regularizacion de Tikhonov). Analizaremos su formulacion matematica, demostraremos por que su solucion matricial resuelve el problema del condicionamiento numerico en presencia de multicolinealidad, trazaremos la ruta de regularizacion (regularization path) y utilizaremos validacion cruzada eficiente con `RidgeCV` para encontrar el hiperparametro optimo $\alpha$.

---

### Contenidos
1. Formulacion Matematica de la Regresion Ridge
2. Solucion Analitica Cerrada y Estabilidad Numerica
3. Efecto del Hiperparametro $\alpha$ y Encogimiento (Shrinkage)
4. Trazado del Camino de Regularizacion (Regularization Path)
5. Seleccion del $\alpha$ Optimo mediante `RidgeCV`
6. Evaluacion y Comparativa frente a OLS en el Dataset Diabetes
7. Analisis de Residuos
8. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Formulacion Matematica de la Regresion Ridge

La regresion Ridge modifica la funcion de costo de minimos cuadrados anadiendo una penalizacion proporcional al cuadrado de la norma $L_2$ del vector de coeficientes:

$$J_{\text{Ridge}}(\beta) = \frac{1}{2n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 + \frac{\alpha}{2} \sum_{j=1}^p \beta_j^2 = \frac{1}{2n} \|y - X\beta\|_2^2 + \frac{\alpha}{2} \|\beta\|_2^2$$

Donde:
- $n$: Numero de muestras.
- $p$: Numero de variables predictoras.
- $\alpha \ge 0$: Hiperparametro de regularizacion que controla el balance entre el ajuste a los datos de entrenamiento y la penalizacion sobre la magnitud de los parametros.
  - Si $\alpha = 0$: La funcion de costo equivale a la regresion lineal ordinaria (OLS).
  - Si $\alpha \to \infty$: La penalizacion domina completamente, forzando todos los coeficientes $\beta_j \to 0$.

*Nota importante*: El intercepto $\beta_0$ no se incluye en el termino de penalizacion $\sum_{j=1}^p \beta_j^2$.


## 2. Solucion Analitica Cerrada y Estabilidad Numerica

Calculando el gradiente de la funcion de costo e igualando a cero:

$$\nabla_\beta J_{\text{Ridge}}(\beta) = -\frac{1}{n} X^T (y - X\beta) + \alpha \beta = 0$$

Despejando $\beta$, obtenemos la solucion matricial cerrada de Ridge:

$$\hat{\beta}_{\text{Ridge}} = (X^T X + n\alpha I)^{-1} X^T y$$

*(En la implementacion de Scikit-Learn con formulacion estandar, se expresa como $(X^T X + \alpha I)^{-1} X^T y$)*.

### ¿Por que esto resuelve la multicolinealidad?
- Si dos variables son colineales, los autovalores de $X^T X$ pueden ser cercanos a cero, haciendo que la matriz sea casi no invertible (mal condicionada).
- Al sumar $\alpha I$, sumamos una constante positiva $\alpha$ a todos los autovalores de la matriz:
  $$\lambda_j(X^T X + \alpha I) = \lambda_j(X^T X) + \alpha > 0$$
- Esto garantiza estrictamente que la matriz es invertible y bien condicionada, eliminando las oscilaciones descontroladas de los coeficientes.


## 3. Preparacion de Datos y Carga

Cargamos el dataset de diabetes en su formato original y realizamos la particion en entrenamiento (80%) y prueba (20%).


In [ ]:
# Carga de datos
diabetes = load_diabetes(scaled=False, as_frame=True)
X = diabetes.frame.drop(columns=['target'])
y = diabetes.frame['target']

# Particion estratificada aleatoria controlada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Dimensiones de entrenamiento: {X_train.shape}")
print(f"Dimensiones de prueba:        {X_test.shape}")


## 4. Trazado del Camino de Regularizacion (Regularization Path)

Para comprender el comportamiento del encogimiento (shrinkage), ajustaremos multiples modelos Ridge a lo largo de un espectro logaritmico de valores de $\alpha$ (desde $10^{-3}$ hasta $10^5$) y graficaremos la trayectoria de cada coeficiente.


In [ ]:
# Definicion de una malla logaritmica de valores de alpha
alphas = np.logspace(-3, 5, 200)

# Estandarizacion previa de caracteristicas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

coeficientes_ridge = []

for a in alphas:
    modelo_ridge = Ridge(alpha=a)
    modelo_ridge.fit(X_train_scaled, y_train)
    coeficientes_ridge.append(modelo_ridge.coef_)

coeficientes_ridge = np.array(coeficientes_ridge)

# Grafico de la ruta de regularizacion
plt.figure(figsize=(11, 6))
for j in range(X.shape[1]):
    plt.plot(alphas, coeficientes_ridge[:, j], label=X.columns[j], linewidth=1.8)

plt.xscale('log')
plt.xlabel('Hiperparametro alpha (Escala Logaritmica)', fontsize=11, fontweight='bold')
plt.ylabel('Magnitud de Coeficientes Estandarizados', fontsize=11, fontweight='bold')
plt.title('Camino de Regularizacion de Ridge: Encogimiento hacia Cero', fontsize=13, fontweight='bold')
plt.axhline(0, color='black', linestyle='--', alpha=0.6)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left", title="Variables")
plt.tight_layout()
plt.show()


## 5. Seleccion del $\alpha$ Optimo mediante `RidgeCV`

Scikit-Learn provee la clase `RidgeCV`, la cual implementa validacion cruzada eficiente para seleccionar el mejor valor de $\alpha$ sin necesidad de crear bucles manuales de `GridSearchCV`.

Configuraremos `cv=5` (5 particiones de validacion cruzada) y evaluaremos sobre una malla de 100 valores de $\alpha$.


In [ ]:
# Evaluacion con RidgeCV dentro de un Pipeline
grid_alphas = np.logspace(-2, 4, 150)

pipeline_ridge_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge_cv', RidgeCV(alphas=grid_alphas, cv=5, scoring='neg_mean_squared_error'))
])

pipeline_ridge_cv.fit(X_train, y_train)

# Extraccion del mejor alpha seleccionado
mejor_alpha = pipeline_ridge_cv.named_steps['ridge_cv'].alpha_
print(f"Mejor hiperparametro alpha seleccionado por CV: {mejor_alpha:.4f}")


## 6. Evaluacion y Comparativa frente a OLS

Comparamos el modelo Ridge ajustado con el $\alpha$ optimo frente al modelo lineal ordinario (OLS) en el conjunto de prueba independiente.


In [ ]:
# 1. Modelo OLS de referencia
pipeline_ols = Pipeline([
    ('scaler', StandardScaler()),
    ('ols', LinearRegression())
])
pipeline_ols.fit(X_train, y_train)

# 2. Predicciones en conjunto de prueba
y_pred_ols = pipeline_ols.predict(X_test)
y_pred_ridge = pipeline_ridge_cv.predict(X_test)

# 3. Metricas
metricas = {
    'Modelo': ['OLS (LinearRegression)', f'Ridge (alpha={mejor_alpha:.2f})'],
    'MSE_Test': [mean_squared_error(y_test, y_pred_ols), mean_squared_error(y_test, y_pred_ridge)],
    'RMSE_Test': [np.sqrt(mean_squared_error(y_test, y_pred_ols)), np.sqrt(mean_squared_error(y_test, y_pred_ridge))],
    'MAE_Test': [mean_absolute_error(y_test, y_pred_ols), mean_absolute_error(y_test, y_pred_ridge)],
    'R2_Test': [r2_score(y_test, y_pred_ols), r2_score(y_test, y_pred_ridge)]
}

df_metricas = pd.DataFrame(metricas)
print("Tabla Comparativa de Rendimiento:")
print(df_metricas.round(4).to_string(index=False))


In [ ]:
# Comparacion de coeficientes: OLS vs Ridge
coef_ols = pipeline_ols.named_steps['ols'].coef_
coef_ridge = pipeline_ridge_cv.named_steps['ridge_cv'].coef_

df_coef_comp = pd.DataFrame({
    'Variable': X.columns,
    'OLS': coef_ols,
    'Ridge': coef_ridge
})

x_idx = np.arange(len(X.columns))
width = 0.35

plt.figure(figsize=(12, 5))
plt.bar(x_idx - width/2, df_coef_comp['OLS'], width=width, label='OLS', color='#d95f02')
plt.bar(x_idx + width/2, df_coef_comp['Ridge'], width=width, label=f'Ridge (alpha={mejor_alpha:.1f})', color='#2b5c8f')

plt.xticks(x_idx, X.columns, fontsize=11)
plt.ylabel('Valor del Coeficiente Estandarizado', fontsize=11, fontweight='bold')
plt.title('Comparacion de Coeficientes Estandarizados: OLS vs. Ridge', fontsize=13, fontweight='bold')
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Calculo de la norma L2 de ambos vectores de coeficientes
norma_ols = np.linalg.norm(coef_ols, ord=2)
norma_ridge = np.linalg.norm(coef_ridge, ord=2)
print(f"Norma L2 del vector de coeficientes OLS:   {norma_ols:.3f}")
print(f"Norma L2 del vector de coeficientes Ridge: {norma_ridge:.3f}")
print(f"Reduccion porcentual en la norma L2:      {((norma_ols - norma_ridge) / norma_ols) * 100:.2f}%")


## 7. Analisis de Residuos

Examinamos los residuos del modelo Ridge en el conjunto de prueba para verificar que no existan patrones no lineales obvios ni heterocedasticidad severa.


In [ ]:
residuos_ridge = y_test - y_pred_ridge

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grafico 1: Residuos vs Predicciones
axes[0].scatter(y_pred_ridge, residuos_ridge, color='#2b5c8f', alpha=0.7, edgecolor='k', s=45)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Valores Predichos ($\\hat{y}$)', fontsize=11)
axes[0].set_ylabel('Residuos ($y - \\hat{y}$)', fontsize=11)
axes[0].set_title('Residuos vs. Predicciones (Ridge)', fontsize=12, fontweight='bold')

# Grafico 2: Histograma de Residuos
sns.histplot(residuos_ridge, kde=True, ax=axes[1], color='#2b5c8f')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Valor del Residuo', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].set_title('Distribucion de Residuos (Ridge)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Conclusiones

### Hallazgos Principales:
1. **Solucion Analitica Estable**: Ridge anade $\alpha I$ a la matriz de covarianza de caracteristicas, garantizando siempre su invertibilidad incluso cuando existen variables con alta colinealidad como `s1` y `s2`.
2. **Encogimiento Continuo**: A medida que $\alpha$ se incrementa, todos los coeficientes disminuyen de magnitud gradualmente, pero ninguno llega a anularse de forma exacta.
3. **Control de la Varianza**: Al reducir la norma $L_2$ global de los coeficientes, el modelo es menos vulnerable a perturbaciones aleatorias en los datos de entrada, mejorando el error cuadratico en datos de prueba.
4. **Limitacion de Ridge**: Al no llevar coeficientes a cero absoluto, Ridge no realiza seleccion automatica de variables; el modelo final sigue dependiendo de las 10 caracteristicas.

En el siguiente cuaderno, **03_Regresion_Lasso_L1_y_Seleccion_Variables.ipynb**, introduciremos la penalizacion $L_1$ (Lasso), capaz de anular coeficientes irrelevantes para generar modelos parsimoniosos e interpretables.
